# Extract word embeddings

Embed the `item` column of each `design_data_*.csv` with **Qwen/Qwen3-Embedding-8B** (4096-dim).

Reads the three design files in `Materials/`, embeds each item, and writes one output CSV per dataset (`embeddings_<cat>_<MODEL_TAG>.csv`) carrying the `ID`/`item`/`crit`/`training` metadata plus 4096 embedding columns.

In [1]:
from pathlib import Path
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

C:\Users\dizydorc\OneDrive\University\Project Github Repositories\Estimation processes in real-world domains\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Setup
FILES = {
    "mammals":   "design_data_mammals.csv",
    "countries": "design_data_countries.csv",
    "food":      "design_data_food.csv",
}

MODEL_NAME = "Qwen/Qwen3-Embedding-8B"  # 4096-dim
MODEL_TAG  = "qwen3"                    # used in output file names
KEEP_COLS  = ["ID", "item", "crit", "training"]   

# Context to disambiguate bare words (e.g. "bat" the mammal vs. baseball bat).
# {item} is the item, {cat} the singular category (see SINGULAR).
# Set to "{item}" for no context.
CONTEXT_TEMPLATE = "the {cat} {item}"
SINGULAR         = {"mammals": "mammal", "countries": "country", "food": "food"}

In [3]:
# First run downloads the model (~16 GB) to the Hugging Face cache; then offline.
# bfloat16 halves memory (~16 GB RAM); left padding as recommended for Qwen3's last-token pooling.
model = SentenceTransformer(
    MODEL_NAME,
    model_kwargs     = {"dtype": torch.bfloat16},
    tokenizer_kwargs = {"padding_side": "left"},
)

The `tokenizer_kwargs` argument was renamed and is now deprecated. Please use `processor_kwargs` instead.
C:\Users\dizydorc\OneDrive\University\Project Github Repositories\Estimation processes in real-world domains\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dizydorc\.cache\huggingface\hub\models--Qwen--Qwen3-Embedding-8B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/window

In [4]:
for cat, fname in FILES.items():
    df = pd.read_csv("../../Materials/" + fname, sep=";", decimal=",", encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]

    items = df["item"].astype(str).str.strip()
    texts = [CONTEXT_TEMPLATE.format(item=w, cat=SINGULAR[cat]) for w in items]
    print(texts)
    vecs = model.encode(
        texts,
        normalize_embeddings = True,
        batch_size           = 32,
        show_progress_bar    = True,
    )

    emb = pd.DataFrame(vecs, columns=[f"emb_{i+1}" for i in range(vecs.shape[1])])
    out = pd.concat([df[KEEP_COLS].reset_index(drop=True), emb], axis=1)

    out_path = f"../../Data/Embeddings/embeddings_{cat}_{MODEL_TAG}.csv"
    out.to_csv(out_path, index=False) 
    print(f"{cat:>9}: {out.shape[0]} items -> {out.shape[1] - len(KEEP_COLS)} dims  "
          f"({out_path})")

['the mammal Aoudad', 'the mammal Brazilian tapir', 'the mammal American bison', 'the mammal Giant anteater', 'the mammal Ibex', "the mammal Thomson's gazelle", 'the mammal Llama', "the mammal Kirk's dik-dik", 'the mammal Red panda', 'the mammal African buffalo', 'the mammal Bactrian camel', 'the mammal Dromedary', 'the mammal Mule deer', 'the mammal Caribou and reindeer', 'the mammal Giraffe', 'the mammal Hippopotamus', 'the mammal Wart hog', 'the mammal Wild boar', 'the mammal Gray wolf', 'the mammal African wild dog', 'the mammal Raccoon dog', 'the mammal Bat-eared fox', 'the mammal Cheetah', 'the mammal Lion', 'the mammal Leopard', 'the mammal Tiger', 'the mammal Dwarf mongoose', 'the mammal Meerkat', 'the mammal Spotted hyena', 'the mammal Sea otter', 'the mammal Least weasel', 'the mammal Steller sea lion', 'the mammal Harbor seal', 'the mammal Common raccoon', 'the mammal Brown bear', 'the mammal Polar bear', 'the mammal Humpback whale', 'the mammal Bottlenosed dolphin', 'the ma

Batches: 100%|██████████| 3/3 [01:43<00:00, 34.63s/it]


  mammals: 80 items -> 4096 dims  (../../Data/Embeddings/embeddings_mammals_qwen3.csv)
['the country China', 'the country India', 'the country United States', 'the country Nigeria', 'the country Brazil', 'the country Russia', 'the country Mexico', 'the country Ethiopia', 'the country Egypt', 'the country Vietnam', 'the country Democratic Republic of the Congo', 'the country Germany', 'the country Iran', 'the country France', 'the country South Africa', 'the country Tanzania', 'the country Colombia', 'the country Argentina', 'the country Canada', 'the country Ukraine', 'the country Angola', 'the country Peru', 'the country Yemen', 'the country Ivory Coast', 'the country Nepal', 'the country Venezuela', 'the country Australia', 'the country Taiwan', 'the country Syria', 'the country Burkina Faso', 'the country Chile', 'the country Netherlands', 'the country Ecuador', 'the country Benin', 'the country Bolivia', 'the country Papua New Guinea', 'the country Haiti', 'the country Cuba', 'the 

Batches: 100%|██████████| 3/3 [01:15<00:00, 25.03s/it]


countries: 80 items -> 4096 dims  (../../Data/Embeddings/embeddings_countries_qwen3.csv)
['the food Spaghetti', 'the food Rice', 'the food Rigatoni', 'the food Bread roll', 'the food Oat flakes', 'the food Couscous', 'the food Bread', 'the food Croissant', 'the food White toast', 'the food Bulgur', 'the food Wrap', 'the food Potatoes', 'the food Cucumber', 'the food Bell pepper', 'the food Carrots', 'the food Corn', 'the food Fried egg', 'the food Broccoli', 'the food Tomatoes', 'the food Iceberg lettuce', 'the food Cauliflower', 'the food Mushrooms', 'the food Peas', 'the food Lentils', 'the food Beans', 'the food Banana', 'the food Apple', 'the food Watermelon', 'the food Kidney beans', 'the food Orange', 'the food Mandarin', 'the food Mango', 'the food Pineapple', 'the food Strawberry', 'the food Raspberry', 'the food Pear', 'the food Apricot', 'the food Grapes', 'the food Yogurt', 'the food Curd cheese', 'the food Gouda cheese', 'the food Butter', 'the food Feta', 'the food Mozzare

Batches: 100%|██████████| 3/3 [01:10<00:00, 23.36s/it]


     food: 80 items -> 4096 dims  (../../Data/Embeddings/embeddings_food_qwen3.csv)
